In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://rdm.example.com/'
admin_rdm_url = 'https://admin.rdm.example.com/'
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
idp_user_display_name_1 = None  # 最近の活動のログ文言に出る表示名 (IdPユーザーのfullname)
idp_name_integrated_admin = None
idp_username_integrated_admin = None
idp_password_integrated_admin = None

# 実行対象ストレージ (必須。STORAGE_REGISTRY のキーのいずれか)
# 'osfstorage' / 's3' / 's3compat' / 's3compatsigv4' / 'dropbox' / 'googledrive' / 'onedrive' / 'nextcloud'
target_storage_id = None

# 's3' / 's3compat' / 's3compatsigv4'
s3_access_key_1 = None
s3_secret_access_key_1 = None
s3_test_bucket_name_1 = None
s3compat_type_name_1 = None

# 'dropbox' / 'googledrive' / 'onedrive' / 'nextcloud'
folder_1 = None

default_result_path = None
close_on_fail = False
transition_timeout = 60000
delete_project = True
has_onlyoffice = False  # True: ONLYOFFICEが利用可能な環境でのみ No.26〜34 を実行する

In [ ]:
import yaml
if target_storage_id in ['dropbox', 'googledrive', 'onedrive', 'nextcloud']:
    config_path = input(prompt='設定ファイルのパス:')
    with open(config_path) as f:
        for k, v in yaml.load(f.read(), Loader=yaml.SafeLoader).items():
            locals()[k] = v

# GakuNinRDM 総合テスト [ストレージ共通-ダウンロード履歴表示機能]

- サブシステム名: アドオン
- ページ/アドオン: 対象ストレージ（`target_storage_id` で指定）
- 機能分類: ダウンロード履歴表示機能
- シナリオ名: 新規ファイルダウンロード / フォルダZIPダウンロード / ONLYOFFICEで開く操作の最近の活動ログ確認（プロジェクトダッシュボード）
- 用意するテストデータ: URL一覧、アカウント(統合管理者, 既存ユーザー1: GRDM)、テスト対象ストレージ設定
- 備考: ダウンロード履歴表示機能検証用の専用notebook。`target_storage_id` でストレージを1つ選択して実行する（NII / S3 / S3 Compatible Storage / S3 Compatible Storage(SigV4) / Dropbox / Google Drive / OneDrive / Nextcloud の8種に対応）。プロジェクトは常に1つ (`rdm_project_name`)。


In [ ]:
STORAGE_REGISTRY = {
    'osfstorage': {'name': 'NII Storage', 'is_auto_setting_storage': True, 'requires': []},
    's3': {'name': 'Amazon S3', 'is_auto_setting_storage': True, 'requires': ['s3_access_key_1', 's3_secret_access_key_1', 's3_test_bucket_name_1']},
    's3compat': {'name': 'S3 Compatible Storage', 'is_auto_setting_storage': True, 'requires': ['s3_access_key_1', 's3_secret_access_key_1', 's3_test_bucket_name_1', 's3compat_type_name_1']},
    's3compatsigv4': {'name': 'S3 Compatible Storage (SigV4)', 'is_auto_setting_storage': True, 'requires': ['s3_access_key_1', 's3_secret_access_key_1', 's3_test_bucket_name_1', 's3compat_type_name_1']},
    'dropbox': {'name': 'Dropbox', 'is_auto_setting_storage': False, 'requires': ['folder_1']},
    'googledrive': {'name': 'Google Drive', 'is_auto_setting_storage': False, 'requires': ['folder_1']},
    'onedrive': {'name': 'OneDrive', 'is_auto_setting_storage': False, 'requires': ['folder_1']},
    'nextcloud': {'name': 'Nextcloud', 'is_auto_setting_storage': False, 'requires': ['folder_1']},
}

if target_storage_id not in STORAGE_REGISTRY:
    raise ValueError(f'target_storage_id must be one of {list(STORAGE_REGISTRY)}, got {target_storage_id!r}')

storage_info = STORAGE_REGISTRY[target_storage_id]
target_storage_name = storage_info['name']
is_auto_setting_storage = storage_info['is_auto_setting_storage']

rdm_project_name = 'TEST-ダウンロード履歴-{}-{}'.format(
    target_storage_name.upper().replace(' ', ''), datetime.now().strftime('%Y%m%d-%H%M%S')
)

missing = [name for name in storage_info['requires'] if globals().get(name) is None]
if missing:
    raise ValueError(f"target_storage_id={target_storage_id!r} requires: {missing}")

(target_storage_id, target_storage_name, is_auto_setting_storage, s3_test_bucket_name_1)


In [ ]:
if idp_name_1 is None:
    idp_name_1 = input(prompt='IdP Name for user 1: ')
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if idp_user_display_name_1 is None:
    idp_user_display_name_1 = input(prompt=f'Display name for {idp_username_1}@{idp_name_1}')

In [ ]:
if idp_name_integrated_admin is None:
    idp_name_integrated_admin = input(prompt=f'IdP Name for integrated admin: ')
if idp_username_integrated_admin is None:
    idp_username_integrated_admin = input(prompt=f'Username for {idp_name_integrated_admin} (integrated admin)')
if idp_password_integrated_admin is None:
    idp_password_integrated_admin = getpass(prompt=f'Password for {idp_username_integrated_admin}@{idp_name_integrated_admin}')


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import asyncio
import importlib
import os
import traceback

import scripts.playwright
importlib.reload(scripts.playwright)
from scripts.playwright import *
from scripts import grdm
importlib.reload(grdm)

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


## ウェブブラウザでGakuNin RDM管理者のトップページを表示する

管理者トップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ログイン情報を用いてスーパー管理者権限を持つユーザーでログインする

個人の管理者ページが表示されること

In [ ]:
async def _step(page):
    await grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin, idp_password_integrated_admin, transition_timeout=transition_timeout
    )
    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「https://{admin_domain}/admin/waffle/switch/」にアクセスする

・「変更する switch を選択」画面が表示されること   
・「NAME」が「enable_download_history_log」の「ACTIVE」が無効になっていること   
※ 前回の実行でNo.39（「enable_download_history_log」の「ACTIVE」を無効に戻す処理）まで完了しなかった場合は、管理者画面の「enable_download_history_log」の「ACTIVE」 を無効に戻してから実行すること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url.rstrip('/') + '/admin/waffle/switch/')
    await expect(page.locator("#content h1")).to_have_text("変更する switch を選択")

    row = page.locator("#result_list tbody tr").filter(has=page.locator("th.field-name a", has_text="enable_download_history_log"))
    await expect(row.locator("td.field-active img")).to_have_attribute("alt", "False")

await run_pw(_step)

## 「NAME」が「enable_download_history_log」のチェックボックスにチェックを入れ、ドロップダウンボックスで「Enable the selected switches.」を選択し、「移動」ボタンをクリックする

「NAME」が「enable_download_history_log」の「ACTIVE」が有効になっていること

In [ ]:
async def _step(page):
    row = page.locator("#result_list tbody tr").filter(has=page.locator("th.field-name a", has_text="enable_download_history_log"))
    await row.locator("input.action-select").check()
    await page.locator("select[name='action']").select_option(label="Enable the selected switches.")
    await page.locator("button[name='index']").click()
    await expect(row.locator("td.field-active img")).to_have_attribute("alt", "True")

await run_pw(_step)

## ウェブブラウザでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await expect(page.locator('//button[text() = "同意する"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## 「GakuNin RDM IdP」(`idp_name_1`) を利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## 以下の手順を実施する・ユーザーメニューから「設定」を選択する・「アドオンアカウント構成」を選択する (`osfstorage` 以外)

「アドオンアカウント構成」画面が表示され、対象ストレージの既存接続の有無が確認できること

In [ ]:
has_connections = False

async def _step(page):
    if target_storage_id == 'osfstorage':
        return
    global has_connections
    await page.locator('//div[@class = "nav-profile-name"]').click()
    await page.locator('//*[@data-analytics-name="Settings"]').click()
    await expect(page.locator('//*[text() = "アドオンアカウント構成"]')).to_be_visible(timeout=transition_timeout)

    await page.locator('//*[text() = "アドオンアカウント構成"]').click()
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../*[text() = "{target_storage_name}"]')).to_be_visible(timeout=transition_timeout)
    await page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../*[text() = "{target_storage_name}"]').scroll_into_view_if_needed()
    try:
        await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]')).to_be_visible(timeout=transition_timeout)
        has_connections = True
    except:
        print('Ignored (no existing connection)')
        traceback.print_exc()
        has_connections = False

await run_pw(_step)

## (「アカウントを切断」が対象ストレージに表示されている場合) すべての「アカウントを切断」をクリック、「アカウントを切断しますか？」のダイアログにて「切断」をクリックする

対象ストレージの既存の接続が残っていないこと（もともと接続がない場合はそのまま次の手順へ進めること）

In [ ]:
async def _step(page):
    if target_storage_id == 'osfstorage' or not has_connections:
        return
    while True:
        print('Disconnecting...')
        await page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]').click()
        await page.locator('.bootbox-confirm .btn-danger').click()
        try:
            await asyncio.sleep(3)
            await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]')).to_be_visible(timeout=transition_timeout)
        except:
            print('Ignored')
            traceback.print_exc()
            break
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 左上 GakuNin RDMロゴをクリックする

既存接続の解除有無によらず、ダッシュボードへ戻り「プロジェクト管理者」の表示が確認できること

In [ ]:
async def _step(page):
    if target_storage_id == 'osfstorage':
        return
    await page.locator('//*[contains(@class, "service-name")]//a').click()
    await expect(page.locator('//*[text() = "プロジェクト管理者"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (プロジェクト一覧に当該プロジェクト名が無い場合) 「新規プロジェクト作成」から新規プロジェクトを作成する

ダッシュボードのプロジェクト一覧に入力したプロジェクト名のプロジェクトが追加されること（既に同名のプロジェクトが存在する場合は作成せず、そのプロジェクトを使用する）

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)
    await grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示され、「アドオン」タブが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトの「アドオン」タブで対象ストレージのアドオンを有効化する (`osfstorage` 以外)

対象ストレージの「有効にする」をクリックし、確認ダイアログを承認する。アドオンが有効化され、「アカウントに接続する」リンク (Google Drive はアカウント認証用リンク) が表示されること

In [ ]:
async def _step(page):
    if target_storage_id == 'osfstorage':
        return
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator(f'//div[@full_name = "{target_storage_name}"]//descendant::a[text() = "有効にする"]')).to_be_visible(timeout=transition_timeout)
    await page.locator(f'//div[@full_name = "{target_storage_name}"]//descendant::a[text() = "有効にする"]').click()
    await expect(page.locator(f'//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)
    await page.locator(f'//button[@data-bb-handler = "confirm"]').click()

    if target_storage_id == 'googledrive':
        await expect(page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(@class, "addon-auth")]')).to_be_visible(timeout=transition_timeout)
    else:
        await expect(page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "アカウントに接続する")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージのアカウントを接続する (`osfstorage` 以外)

自動接続系 (Amazon S3 / S3 Compatible Storage / 同 SigV4) はアクセスキー等を入力し、テスト用バケットを選択して「保存」すると「正常にリンクされました」が表示されること。   
OAuth系（Dropbox / Google Drive / OneDrive / Nextcloud）は、表示されたURLをシークレットウィンドウで開き、ストレージアカウントの認証を完了した後、「finished」と入力できること。また、プロジェクトのアドオン設定画面に「変更」ボタンが表示されること。

In [ ]:
async def _step(page):
    if target_storage_id == 'osfstorage':
        return

    if is_auto_setting_storage:
        await page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/..//a[contains(text(), "アカウントに接続する")]').click()
        await expect(page.locator(f'#{target_storage_id}InputCredentials .btn-success')).to_be_enabled(timeout=transition_timeout)
        await asyncio.sleep(1)

        if target_storage_id in ('s3compat', 's3compatsigv4'):
            await page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//select[@id = "selected_service"]').select_option(s3compat_type_name_1)
            access_key_field = page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//input[@name = "access_key"]')
            secret_access_key_field = page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//input[@name = "secret_key"]')
        else:
            access_key_field = page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//input[@id = "access_key"]')
            secret_access_key_field = page.locator(f'//*[@id = "{target_storage_id}InputCredentials"]//input[@id = "secret_key"]')
        await access_key_field.fill(s3_access_key_1)
        await secret_access_key_field.fill(s3_secret_access_key_1)

        await page.locator(f'#{target_storage_id}InputCredentials .btn-success').click()
        await expect(page.locator(f'//button[text() = "接続"]')).to_be_enabled(timeout=transition_timeout)
        await asyncio.sleep(1)
        await page.locator(f'//button[text() = "接続"]').click()
        await expect(page.locator(f'//button[@id = "newBucket"]')).to_be_visible(timeout=transition_timeout)

        await page.locator(f'//span[text() = "{s3_test_bucket_name_1}"]/../../..//input[@type = "radio"]').click()

        await asyncio.sleep(1)
        await page.locator(f'//*[@class = "{target_storage_id}-confirm-selection"]//input[@value = "保存"]').click()
        await expect(page.locator(f'//*[@id = "{target_storage_id}Scope"]/*[contains(@class, "help-block")]//*[contains(text(), "正常にリンクされました")]')).to_be_visible(timeout=transition_timeout)
    else:
        print(page.url)
        confirmed = input(
            prompt='上のURLをシークレットウィンドウで開き、IdPアカウントでログイン→「アカウントに接続する」をクリック→ストレージアカウントで認証してください。完了したら finished と入力してください。'
        )
        assert confirmed.strip().lower() == 'finished'
        await page.reload()

        await page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "変更"]').scroll_into_view_if_needed()
        await expect(page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "変更"]')).to_be_visible(timeout=transition_timeout)
        await asyncio.sleep(1)
        
await run_pw(_step)

## (OAuth系のみ) 連携するフォルダを選択する

ページをリロードし「変更」→連携するフォルダのラジオボタン→「保存」の順に操作し、「正常にリンクされました」のメッセージが表示されること (自動接続系と `osfstorage` は本セルをスキップ)

In [ ]:
async def _step_after_oauth(page):
    if target_storage_id == 'osfstorage' or is_auto_setting_storage:
        return

    await page.reload()
    await expect(page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "変更"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)
    await page.locator(f'//img[@src = "/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "変更"]').click()
    await expect(page.locator(f'//span[text() = "{folder_1}"]/../../..//input[@type = "radio"]')).to_be_enabled(timeout=transition_timeout)
    await asyncio.sleep(1)
    await page.locator(f'//span[text() = "{folder_1}"]/../../..//input[@type = "radio"]').click()

    await asyncio.sleep(1)
    await page.locator(f'//*[@class = "{target_storage_id}-confirm-selection"]//input[@value = "保存"]').click()
    await expect(page.locator(f'//*[@id = "{target_storage_id}Scope"]/*[contains(@class, "help-block")]//*[contains(text(), "正常にリンクされました")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step_after_oauth)

## 「ファイルページ」リンクをクリックし、対象ストレージが表示されることを確認する (`osfstorage` 以外)

対象ストレージがファイル一覧に展開表示されること

In [ ]:
async def _step(page):
    if target_storage_id == 'osfstorage':
        return
    try:
        await page.get_by_role("link", name="ファイルページ").click()
    except:
        print('Ignored')
        traceback.print_exc()
        await page.locator('//*[@id = "projectNavFiles"]//a').click()
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
await run_pw(_step)

## プロジェクト概要ページへ戻り、対象ストレージを確認して「最近の活動」パネルを展開する

`osfstorage` 以外は画面上部のプロジェクト名リンクで概要ページへ戻る (`osfstorage` は既に概要ページにいるため戻る操作は不要)。
プロジェクトダッシュボードに「アドオン」タブおよび対象ストレージが表示されており、「最近の活動」パネルが展開されること

In [ ]:
async def _step(page):
    if target_storage_id != 'osfstorage':
        # 「ファイルページ」経由で離脱している場合はプロジェクト概要ページへ戻る
        await page.locator('a.project-title').click()

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(1)

    # 「最近の活動」パネルを開いておく（以降の verify_download_log で #logFeed を参照する）
    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## 上部メニューの「アップロード」からストレージへ新規ファイルをアップロードする

ファイル一覧の対象ストレージの下のツリーにファイルが追加されること（ファイルは "x" で埋められた1kBのファイルとする）

In [ ]:
yyyymmdd = datetime.now().strftime('%Y%m%d')
filename = f'{yyyymmdd}_ダウンロード履歴テスト_1kB.txt'
filepath = os.path.join(work_dir, filename)

with open(filepath, 'w') as f:
    for i in range(1000):
        f.write('x')

async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, filepath)
    await asyncio.sleep(2)

    await expect(page.locator(f'//*[text() = "{filename}"]/../following-sibling::*//*[@role = "progressbar"]')).to_have_count(0, timeout=transition_timeout)
    await expect(grdm.get_select_file_title_locator(page, filename)).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧から「ファイル(1kB)」の行のファイル名以外の部分をクリックする

ファイル一覧上部のファイル操作メニューが表示され、「ダウンロード」が有効になること

In [ ]:
async def _step(page):
    await expect(grdm.get_select_file_title_locator(page, filename)).to_be_visible(timeout=transition_timeout)
    await grdm.get_select_file_extension_locator(page, filename).click()
    await expect(page.locator('//i[contains(@class, "fa-download")]/../*[text() = "ダウンロード"]')).to_be_enabled()

await run_pw(_step)

## ファイル一覧の上部メニューから「ダウンロード」をクリックし、ダウンロード完了後にページをリロードして「最近の活動」を確認する

・ファイルがダウンロードされること   
・最近の活動に「{ログインユーザーの表示名}が{作成したプロジェクト名}の{対象ストレージ}からファイルをダウンロードしました」のログが記録されること (ストレージ別の実文言は `grdm.verify_download_log` のテンプレートに準拠)   

In [ ]:
async def _step(page):
    async with page.expect_download(timeout=transition_timeout * 2) as download_info:
        await page.locator('//i[contains(@class, "fa-download")]/../*[text() = "ダウンロード"]').click()
    download = await download_info.value
    download_dir = os.path.join(work_dir, 'downloaded')
    os.makedirs(download_dir, exist_ok=True)
    download_path = os.path.join(download_dir, download.suggested_filename)
    await download.save_as(download_path)

    await page.reload()
    await grdm.verify_download_log(page, idp_user_display_name_1, rdm_project_name, filename, target_storage_id, False, s3_test_bucket_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## ファイル一覧の対象ストレージの行をクリックし、上部メニューから「新規フォルダ作成」をクリックする

ファイル操作メニューがテキスト入力フォームに切り替わること

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await page.locator('//i[contains(@class, "fa-plus")]/../*[text() = "新規フォルダ作成"]').click()
    await expect(page.locator('#createFolderInput')).to_be_editable()

await run_pw(_step)

## テキスト入力フォームに「{yyyymmdd}_ダウンロード履歴テスト_フォルダ1」と入力してEnterキーを押下する

ファイル一覧に「{yyyymmdd}_ダウンロード履歴テスト_フォルダ1」の名称でフォルダが追加されること

In [ ]:
new_foldername = f'{yyyymmdd}_ダウンロード履歴テスト_フォルダ1'

async def _step(page):
    create_folder_input = page.locator('#createFolderInput')
    await create_folder_input.fill(new_foldername)
    await create_folder_input.press('Enter')
    await grdm.get_select_folder_title_locator(page, new_foldername).scroll_into_view_if_needed()
    await expect(grdm.get_select_folder_title_locator(page, new_foldername)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アップロードしたファイルをドラッグ＆ドロップでフォルダへ移動する

ZIP対象のフォルダを中身ありの状態にするための手順。ファイルの表示行が対象ストレージ直下からフォルダの下のツリーへ移動し、「移動に成功しました」のメッセージが表示されること

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, filename).click()
    source = grdm.get_select_file_draggable_locator(page, filename)
    dest = grdm.get_select_folder_droppable_locator(page, new_foldername)

    await grdm.drag_and_drop(page, source, dest)
    await expect(page.locator('//*[contains(text(), "移動に成功しました")]')).to_be_visible(timeout=transition_timeout * 5)

await run_pw(_step)

## ファイル一覧から「{yyyymmdd}_ダウンロード履歴テスト_フォルダ1」の行のファイル名以外の部分をクリックする

ファイル一覧上部のファイル操作メニューが表示され、「ZIPでダウンロード」が有効になること

In [ ]:
async def _step(page):
    await expect(grdm.get_select_folder_title_locator(page, new_foldername)).to_be_visible(timeout=transition_timeout)
    await grdm.get_select_folder_title_locator(page, new_foldername).click()
    await expect(page.locator('//i[contains(@class, "fa-download")]/../*[text() = "ZIPでダウンロード"]')).to_be_enabled()

await run_pw(_step)

## ファイル一覧の上部メニューから「ZIPでダウンロード」をクリックし、ダウンロード完了後にページをリロードして「最近の活動」を確認する

・ZIPファイルにはアップロード済みのファイルが圧縮されていること   
・最近の活動に「{ログインユーザーの表示名}が{作成したプロジェクト名}の{対象ストレージ}からフォルダをZIPとしてダウンロードしました」のログが記録されること (ストレージ別の実文言は `grdm.verify_download_log` のテンプレートに準拠)

In [ ]:
import zipfile

async def _step(page):
    async with page.expect_download() as download_info:
        await page.locator('//i[contains(@class, "fa-download")]/../*[text() = "ZIPでダウンロード"]').click()
    download = await download_info.value
    download_dir = os.path.join(work_dir, 'downloaded')
    os.makedirs(download_dir, exist_ok=True)
    zip_path = os.path.join(download_dir, download.suggested_filename)
    await download.save_as(zip_path)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        assert zip_ref.namelist() == [filename], zip_ref.namelist()

    await page.reload()
    await grdm.verify_download_log(page, idp_user_display_name_1, rdm_project_name, new_foldername, target_storage_id, True, s3_test_bucket_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## 以下の手順を実施し、作成した「{yyyymmdd}_ダウンロード履歴テスト_フォルダ1」を削除する・ファイル一覧から「{yyyymmdd}_ダウンロード履歴テスト_フォルダ1」の行をクリックする・上部メニューから「フォルダを削除」をクリックする・確認ダイアログで削除を実行する

ファイル一覧のツリーから「{yyyymmdd}_ダウンロード履歴テスト_フォルダ1」が消えること
※ テスト用フォルダの残留を避けるための後片付け手順

In [ ]:
async def _step(page):
    await grdm.get_select_folder_title_locator(page, new_foldername).click()
    await page.locator('//i[contains(@class, "fa-trash")]/../*[text() = "フォルダを削除"]').click()
    await page.locator('//*[@id = "tb-tbody"]//*[@class = "modal-content"]//*[contains(@class, "btn-danger")]').scroll_into_view_if_needed()
    await expect(page.locator('//*[@id = "tb-tbody"]//*[@class = "modal-content"]//*[contains(@class, "btn-danger")]')).to_be_enabled(timeout=transition_timeout)
    await page.locator('//*[@id = "tb-tbody"]//*[@class = "modal-content"]//*[contains(@class, "btn-danger")]').click()
    await expect(grdm.get_select_folder_title_locator(page, new_foldername)).to_have_count(0, timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) ファイル一覧の対象ストレージ名の行をクリックする

ファイル一覧上部のファイル操作メニューが表示されること

In [ ]:
async def _step(page):
    if not has_onlyoffice:
        return
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await expect(page.locator('//i[contains(@class, "fa-plus")]/../*[text() = "新規ファイル作成"]')).to_be_enabled()

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) ファイル一覧の上部メニューから「新規ファイル作成」をクリックする

ファイル操作メニューがテキスト入力フォームに切り替わること

In [ ]:
async def _step(page):
    if not has_onlyoffice:
        return
    await page.locator('//i[contains(@class, "fa-plus")]/../*[text() = "新規ファイル作成"]').click()
    await expect(page.locator('#createFileInput')).to_be_editable()

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) テキスト入力フォームに「{yyyymmdd}_ファイル.txt」と入力して、テキストのアイコンを押下する
  
・ONLYOFFICE の編集画面が新しいタブで自動で開くこと   
・ファイル一覧に「{yyyymmdd}_ファイル.txt」の名称でファイルが追加されること   
・最近の活動に「{ログインユーザーの表示名}が{作成したプロジェクト名}の{対象ストレージ}にファイル{ファイル名}を追加しました」のログが記録されること (ストレージ別の実文言は `grdm.verify_add_file_log` のテンプレートに準拠)   
・最近の活動に「{ログインユーザーの表示名}が{作成したプロジェクト名}の{対象ストレージ}のファイル{ファイル名}をONLYOFFICEで開きました」のログが記録されること (ストレージ別の実文言は `grdm.verify_open_onlyoffice_log` のテンプレートに準拠)   

In [ ]:
filename = f"{yyyymmdd}_ファイル.txt"
total_number_open_onlyoffice_log = 0
file_url = None

async def _step(page):
    if not has_onlyoffice:
        return
    await page.locator("#createFileInput").fill(filename)
    async with page.expect_popup() as new_page_info:
        await page.locator("i.file-extension._txt").click()

    new_page = await new_page_info.value
    await new_page.wait_for_load_state("domcontentloaded")
    global file_url
    file_url = new_page.url
    
    await expect(grdm.get_select_file_title_locator(page, filename)).to_have_count(1, timeout=transition_timeout)
    await page.reload()
    await grdm.verify_add_file_log(
        page, idp_user_display_name_1, rdm_project_name, filename,
        target_storage_id, s3_test_bucket_name_1, transition_timeout=transition_timeout
    )
    
    global total_number_open_onlyoffice_log
    total_number_open_onlyoffice_log = total_number_open_onlyoffice_log + 1
    await grdm.verify_open_onlyoffice_log(
        page, idp_user_display_name_1, rdm_project_name, filename,
        target_storage_id, total_number_open_onlyoffice_log, transition_timeout=transition_timeout
    )

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) ファイル一覧から「{yyyymmdd}_ファイル.txt」のファイル名部分をクリックする

ファイルプレビュー画面が表示されること

In [ ]:
async def _step(page):
    if not has_onlyoffice:
        return
    await grdm.get_select_file_title_locator(page, filename).click(timeout=transition_timeout)

    await expect(page.locator("#mfrIframe")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="ONLYOFFICEで編集")).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) ブラウザの戻るをクリックし、「最近の活動」を確認する

・最近の活動に「{ログインユーザーの表示名}が{作成したプロジェクト名}の{対象ストレージ}のファイル{ファイル名}をONLYOFFICEで開きました」のログの件数が1件のままであること（プレビューはONLYOFFICEで開いたログを作成しないこと）   
・ダウンロードログも記録されないこと

In [ ]:
async def _step(page):
    if not has_onlyoffice:
        return
    await page.go_back()

    await grdm.verify_open_onlyoffice_log(
        page, idp_user_display_name_1, rdm_project_name, filename,
        target_storage_id, total_number_open_onlyoffice_log, transition_timeout=transition_timeout
    )
    await grdm.verify_download_log(
        page, idp_user_display_name_1, rdm_project_name, filename,
        target_storage_id, False, s3_test_bucket_name_1,
        expected_count=0, transition_timeout=transition_timeout
    )

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) ファイル一覧から「{yyyymmdd}_ファイル.txt」のファイル名部分をクリックする

ファイルプレビュー画面が表示されること

In [ ]:
async def _step(page):
    if not has_onlyoffice:
        return
    await grdm.get_select_file_title_locator(page, filename).click(timeout=transition_timeout)
    
    await expect(page.locator("#mfrIframe")).to_be_visible(timeout=transition_timeout)
    await expect(page.get_by_role("link", name="ONLYOFFICEで編集")).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) 「ONLYOFFICEで編集」ボタンをクリックする

「{yyyymmdd}_ファイル.txt」の編集画面が新しいタブで開くこと

In [ ]:
from urllib.parse import unquote

async def _step(page):
    if not has_onlyoffice:
        return
    async with page.expect_popup() as new_page_info:
        await page.get_by_role("link", name="ONLYOFFICEで編集").click()

    new_page = await new_page_info.value
    await new_page.wait_for_load_state("domcontentloaded")
    print(unquote(unquote(file_url)))
    print(unquote(unquote(new_page.url)))
    assert unquote(unquote(file_url)) == unquote(unquote(new_page.url))

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) ブラウザの戻るをクリックし、「最近の活動」を確認する

最近の活動に「{ログインユーザーの表示名}が{作成したプロジェクト名}の{対象ストレージ}のファイル{ファイル名}をONLYOFFICEで開きました」のログの件数が2件に増えること (ストレージ別の実文言は `grdm.verify_open_onlyoffice_log` のテンプレートに準拠)

In [ ]:
async def _step(page):
    if not has_onlyoffice:
        return
    await page.go_back()

    global total_number_open_onlyoffice_log
    total_number_open_onlyoffice_log = total_number_open_onlyoffice_log + 1
    await grdm.verify_open_onlyoffice_log(
        page, idp_user_display_name_1, rdm_project_name, filename,
        target_storage_id, total_number_open_onlyoffice_log, transition_timeout=transition_timeout
    )

await run_pw(_step)

## (ONLYOFFICE が利用可能な環境の場合) 以下の手順を実施し、作成した「{yyyymmdd}_ファイル.txt」を削除する・ファイル一覧から「{yyyymmdd}_ファイル.txt」の行のファイル名以外の部分をクリックする・上部メニューから「削除」をクリックする・確認ダイアログで削除を実行する

ファイル一覧のツリーから「{yyyymmdd}_ファイル.txt」が消えること

In [ ]:
async def _step(page):
    if not has_onlyoffice:
        return
    await grdm.get_select_file_extension_locator(page, filename).click()
    await expect(page.locator('//i[contains(@class, "fa-trash")]/../*[text() = "削除"]')).to_be_enabled()
    
    await page.locator('//i[contains(@class, "fa-trash")]/../*[text() = "削除"]').click()
    await expect(page.locator('//*[@id = "tb-tbody"]//*[@class = "modal-content"]//*[contains(@class, "btn-danger")]')).to_be_enabled(timeout=transition_timeout)
    
    await page.locator('//*[@id = "tb-tbody"]//*[@class = "modal-content"]//*[contains(@class, "btn-danger")]').click()
    await expect(grdm.get_select_file_title_locator(page, filename)).to_have_count(0, timeout=transition_timeout)
    await asyncio.sleep(1)

await run_pw(_step)

## (指定がある場合)  以下の手順を実施し、プロジェクトを削除する・プロジェクトダッシュボードの上部メニューから「設定」をクリックする・「プロジェクトを削除」を選択し、確認文字列を入力して削除する

以下を確認すること   
・設定画面が表示されること   
・ダッシュボードのプロジェクト一覧からプロジェクトが消えること   
※ STG環境等でのテスト用プロジェクトの残留を避けるための追加手順   

In [ ]:
async def _step(page):
    if not delete_project:
        return
    await grdm.delete_project(page, transition_timeout=transition_timeout)

    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## 対象ストレージのアカウント接続を解除する (`osfstorage` 以外)

対象ストレージの既存接続がすべて解除されていること
※ 次回試験実行時の接続エラーを避けるための後片付け手順

In [ ]:
async def _step(page):
    if target_storage_id == 'osfstorage':
        return

    await asyncio.sleep(1)
    await page.locator('//div[@class = "nav-profile-name"]').click()
    await page.locator('//*[@data-analytics-name="Settings"]').click()
    await expect(page.locator('//*[text() = "アドオンアカウント構成"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//*[text() = "アドオンアカウント構成"]').click()
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../*[text() = "{target_storage_name}"]')).to_be_visible(timeout=transition_timeout)
    await page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../*[text() = "{target_storage_name}"]').scroll_into_view_if_needed()
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]')).to_be_visible(timeout=transition_timeout)

    while True:
        print('Disconnecting...')
        await page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]').click()
        await page.locator('.bootbox-confirm .btn-danger').click()
        try:
            await asyncio.sleep(3)
            await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"][1]')).to_be_visible(timeout=transition_timeout)
        except:
            print('Ignored')
            traceback.print_exc()
            break
    await expect(page.locator(f'//*[@src="/static/addons/{target_storage_id}/comicon.png"]/../..//*[text() = "アカウントを切断"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step) 

## ログイン済みの管理画面へ再度アクセスする

管理者としてログイン済みの状態が維持されており、個人の管理者ページが表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)
    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「https://{admin_domain}/admin/waffle/switch/」にアクセスする

・「変更する switch を選択」画面が表示されること   
・「NAME」が「enable_download_history_log」の「ACTIVE」が有効になっていること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url.rstrip('/') + '/admin/waffle/switch/')
    await expect(page.locator("#content h1")).to_have_text("変更する switch を選択")

    row = page.locator("#result_list tbody tr").filter(has=page.locator("th.field-name a", has_text="enable_download_history_log"))
    await expect(row.locator("td.field-active img")).to_have_attribute("alt", "True")

await run_pw(_step)

## 「NAME」が「enable_download_history_log」のチェックボックスにチェックを入れ、ドロップダウンボックスで「Disable the selected switches.」を選択し、「移動」ボタンをクリックする

「NAME」が「enable_download_history_log」の「ACTIVE」が無効になっていること

In [ ]:
async def _step(page):
    row = page.locator("#result_list tbody tr").filter(has=page.locator("th.field-name a", has_text="enable_download_history_log"))
    await row.locator("input.action-select").check()
    await page.locator("select[name='action']").select_option(label="Disable the selected switches.")
    await page.locator("button[name='index']").click()
    await expect(row.locator("td.field-active img")).to_have_attribute("alt", "False")

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}